# [6.2] Gemma Scope Deep Dive - Exercises

Build the validation ladder for released Gemma Scope-style features: artifact metadata, tagged hypotheses, prompt-level feature scores, held-out AUC, base-vs-instruction deltas, ablation controls, steering guards, and direct logit attribution.

```yaml
gt_tier: GT-1 artifact preflight with GT-0 validation controls
exercise_id: 6.2-gemma-scope-deep-dive
expected_runtime: 45-75 minutes for CPU exercises; several minutes for CUDA Gemma Scope artifact preflight
requires_gpu: true for the artifact preflight; false for the implementation exercises
```

Reading map: review SAE feature activations, ROC AUC, ablation, steering, and direct logit attribution. Failure modes to watch for: treating tags as evidence, not reporting the score reduction, validating only top activations, losing delta signs, ablation without a random control, steering without a perplexity guard, and treating DLA as proof.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter6_sparse_feature_methods"
section = "part2_gemma_scope_deep_dive"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_gemma_scope_deep_dive.tests as tests


@dataclass(frozen=True)
class FeatureArtifactMetadata:
    model_name: str
    artifact_name: str
    artifact_type: Literal["sae", "transcoder"]
    layer: int
    hook_name: str
    d_model: int
    n_features: int


@dataclass(frozen=True)
class TaggedFeatureSpec:
    feature_id: int
    layer: int
    tags: tuple[str, ...]
    description: str


@dataclass(frozen=True)
class FeatureValidationSuiteReport:
    feature_auc: float
    baseline_auc: float
    auc_margin: float
    threshold_accuracy: float
    positive_mean: float
    negative_mean: float
    passes_baseline: bool


@dataclass(frozen=True)
class BaseInstructionFeatureDelta:
    base_mean: float
    instruction_mean: float
    delta: float
    abs_delta: float


@dataclass(frozen=True)
class AblationControlReport:
    baseline_mean: float
    ablated_mean: float
    random_ablated_mean: float
    ablation_delta: float
    random_delta: float
    passes_control: bool


@dataclass(frozen=True)
class SteeringSafetyReport:
    baseline_mean: float
    steered_mean: float
    random_mean: float
    steered_delta: float
    random_delta: float
    perplexity_ratio: float
    passes_control: bool
    passes_perplexity_guard: bool


## Artifact Metadata And Tags

Difficulty: easy. Importance: high. Expected output: complete metadata passes, incomplete metadata fails, and tag filtering preserves feature order. Common bug: treating a prose feature description as validation evidence.


In [ ]:
def metadata_is_complete(metadata: FeatureArtifactMetadata) -> bool:
    raise NotImplementedError()


def features_with_tag(
    features: list[TaggedFeatureSpec],
    tag: str,
) -> list[TaggedFeatureSpec]:
    raise NotImplementedError()


tests.test_metadata_completeness_and_tag_selection(
    FeatureArtifactMetadata,
    TaggedFeatureSpec,
    metadata_is_complete,
    features_with_tag,
)


## Prompt-Level Feature Scores

Difficulty: easy. Importance: high. Expected output: max, mean, and last-token reductions match the controlled tensor, and rank-2 activations are treated as already reduced. Common bug: silently using max reduction everywhere and forgetting to report it.


In [ ]:
def feature_score_vector(
    feature_acts: t.Tensor,
    feature_id: int,
    *,
    reduction: Literal["max", "mean", "last"] = "max",
) -> t.Tensor:
    raise NotImplementedError()


tests.test_feature_score_vector_reductions_match_reference(feature_score_vector)


## Held-Out Feature Validation

Difficulty: medium. Importance: high. Expected output: the candidate feature has AUC `1.0`, the inverted baseline has AUC `0.0`, and the candidate passes the margin check. Common bug: reporting top activating examples without matched negatives.


In [ ]:
def roc_auc_binary(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def validate_feature_scores(
    feature_scores: t.Tensor,
    labels: t.Tensor,
    baseline_scores: t.Tensor,
    *,
    min_auc_margin: float = 0.1,
) -> FeatureValidationSuiteReport:
    raise NotImplementedError()


tests.test_validate_feature_scores_beats_baseline_and_reports_means(
    validate_feature_scores,
    roc_auc_binary,
)


## Base Vs Instruction Deltas

Difficulty: easy. Importance: medium. Expected output: the test checks both positive and negative deltas, preserving sign while reporting absolute magnitude. Common bug: taking `abs` too early and losing the direction of change.


In [ ]:
def base_instruction_feature_delta(
    base_scores: t.Tensor,
    instruction_scores: t.Tensor,
) -> BaseInstructionFeatureDelta:
    raise NotImplementedError()


tests.test_base_instruction_delta_reports_signed_and_abs_change(
    base_instruction_feature_delta,
)


## Ablation Control

Difficulty: medium. Importance: high. Expected output: target ablation reduces the score by `0.75`, random ablation by `0.15`, and the control fails when the roles are reversed. Common bug: checking only that ablation changes the metric, not that it beats random ablation.


In [ ]:
def ablation_control_report(
    baseline_scores: t.Tensor,
    ablated_scores: t.Tensor,
    random_ablated_scores: t.Tensor,
) -> AblationControlReport:
    raise NotImplementedError()


tests.test_ablation_control_requires_target_ablation_to_beat_random(
    ablation_control_report,
)


## Steering Guard

Difficulty: medium. Importance: high. Expected output: useful steering passes when perplexity ratio is `1.1`, and fails the guard when the same target effect comes with a ratio of `1.5`. Common bug: reporting steering success without checking general-model degradation.


In [ ]:
def steering_safety_report(
    baseline_scores: t.Tensor,
    steered_scores: t.Tensor,
    random_control_scores: t.Tensor,
    *,
    baseline_perplexity: float,
    steered_perplexity: float,
    max_perplexity_ratio: float = 1.2,
) -> SteeringSafetyReport:
    raise NotImplementedError()


tests.test_steering_safety_report_checks_control_and_perplexity_guard(
    steering_safety_report,
)


## Direct Logit Attribution

Difficulty: medium. Importance: medium. Expected output: selecting token ids `[0, 2]` returns `[[1.0, 3.0], [4.0, 6.0]]` for the controlled decoder/unembedding pair. Common bug: treating DLA as proof instead of a token-effect hypothesis.


In [ ]:
def direct_logit_attribution(
    decoder_vectors: t.Tensor,
    unembedding: t.Tensor,
    token_ids: t.Tensor | list[int] | None = None,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_direct_logit_attribution_matches_selected_token_projection(
    direct_logit_attribution,
)


## Final Verification

After the implementation cells pass, compare your functions against the reference implementation and then run the artifact preflight from a Python process with CUDA available:

```python
from part2_gemma_scope_deep_dive import solutions
solutions.run_gpu_test(max_vram_gb=24.0)
```

The current checked path validates the pinned Gemma Scope SAE artifact and a semantic feature hypothesis on authenticated real Gemma 3 activations against random-feature and label-shuffle controls.


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
